In [ ]:
import warnings
warnings.simplefilter("ignore", UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import os
import time
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# from models.recon_Update import train_recon
# from models.evaluation import evaluate_recon
from data.subsample import create_mask_for_mask_type
# from tensorboardX import SummaryWriter
# from utils.options import args_parser
# from models.evaluation import test_recon_save
from torch.utils.data import random_split
from data.mri_data import SliceData
from data import transforms
from data import transformsPB as Tpb
#from models.Recurrent_Transformer_new import ReconFormer
import pathlib
from torch.utils.data import DataLoader
from skimage.metrics import structural_similarity as ssim
import fastmri
#from data.combined_GradLoss import DualStreamLoss
from torch.nn import functional as F
# from fastmri.data import transforms as T

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class DataTransform:
    """
    Data Transformer for training U-Net models.
    """

    def __init__(self, resolution, which_challenge, mask_func=None, use_seed=True):
        """
        Args:
            mask_func (common.subsample.MaskFunc): A function that can create a mask of
                appropriate shape.
            resolution (int): Resolution of the image.
            which_challenge (str): Either "singlecoil" or "multicoil" denoting the dataset.
            use_seed (bool): If true, this class computes a pseudo random number generator seed
                from the filename. This ensures that the same mask is used for all the slices of
                a given volume every time.
        """
        if which_challenge not in ('singlecoil', 'multicoil'):
            raise ValueError(
                f'Challenge should either be "singlecoil" or "multicoil"')
        self.mask_func = mask_func
        self.resolution = resolution
        self.which_challenge = which_challenge
        self.use_seed = use_seed

    def __call__(self, kspace, mask, target, attrs, fname, slice):
        """
        Args:
            kspace (numpy.array): Input k-space of shape (num_coils, rows, cols, 2) for multi-coil
                data or (rows, cols, 2) for single coil data.
            mask (numpy.array): Mask from the test dataset
            target (numpy.array): Target image
            attrs (dict): Acquisition related information stored in the HDF5 object.
            fname (str): File name
            slice (int): Serial number of the slice.
        Returns:
            (tuple): tuple containing:
                image (torch.Tensor): Zero-filled input image.
                target (torch.Tensor): Target image converted to a torch Tensor.
                mean (float): Mean value used for normalization.
                std (float): Standard deviation value used for normalization.
        """
        target = Tpb.to_tensor(target.astype(complex)) # (Height x Width x 2)
        kspace = Tpb.fft2c_new(target)                 # (Height x Width x 2)
        
        # Apply mask
        if self.mask_func:
            seed = None if not self.use_seed else tuple(map(ord, fname))
            masked_kspace, mask = transforms.apply_mask(
                kspace, self.mask_func, seed)
        else:
            masked_kspace = kspace

        # Inverse Fourier Transform to get zero filled solution
        image = Tpb.ifft2c_new(masked_kspace) # (Height, Width, 2)

        # Absolute value
        abs_image = Tpb.complex_abs(image) # (height, width)
        mean = torch.tensor(0.0)
        std = abs_image.mean()
        # Normalize input
        image = image.permute(2, 0, 1)    # (2 x Height x Width)
        target = target.permute(2, 0, 1)  # (2 x Height x Width)
        image = transforms.normalize(image, mean, std, eps=0)
        masked_kspace = masked_kspace.permute(2, 0, 1)  # (2 x Height x Width)
        masked_kspace = transforms.normalize(masked_kspace, mean, std, eps=0)
        # Normalize target
        target = transforms.normalize(target, mean, std, eps=0)
        mask = mask.repeat(image.shape[1], 1, 1).squeeze().unsqueeze(0)  # (1 x Height x Width)
        return image, target, mean, std, attrs['norm'].astype(np.float32), fname, slice, attrs['max'].astype(np.float32), mask, masked_kspace

In [ ]:
def _create_dataset(data_paths, transform, batch_size, shuffle, sample_rate, num_workers):
    datasets = [SliceData(root=pathlib.Path(path), transform=transform, sample_rate=sample_rate, challenge='multicoil',sequence='PD') for path in data_paths]
    datasets = torch.utils.data.ConcatDataset(datasets)

    train_ratio = 0.75
    train_size = int(len(datasets) * train_ratio)
    val_size = len(datasets) - train_size

    train_dataset, val_dataset = random_split(datasets, [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return {train_loader, val_loader} #DataLoader(torch.utils.data.ConcatDataset(datasets), batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=True)

In [ ]:
mask_func = create_mask_for_mask_type('random', [0.08], [4])
data_paths = ["./Dataset"]
transform = DataTransform(resolution=320, which_challenge='multicoil', mask_func=mask_func, use_seed=True)
trainloader, valloader = _create_dataset(data_paths=data_paths, transform=transform, batch_size=4, shuffle=True, sample_rate=1.0, num_workers=0)

print("Train Loader:\nRunning 3 iterations with batch size 4...")
for i, batch in enumerate(trainloader):
    if i >= 3:
        break
    print(f"\nBatch {i+1}")
    image, target, mean, std, norm, fname, slice_num, max_val, mask, masked_kspace = batch
    print(f"  image.shape: {image.shape}")
    print(f"  target.shape: {target.shape}")
    print(f"  mask.shape: {mask.shape}")
    print(f"  masked_kspace.shape: {masked_kspace.shape}")
    for j in range(4):
        print(f"  → Slice {slice_num[j]} | File: {fname[j]}")
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        def show_image(tensor_img, title, ax):
            img = tensor_img.numpy()
            img = np.abs(img)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            ax.imshow(img, cmap='gray')
            ax.set_title(title)
            ax.axis('off')
        show_image(torch.sqrt(image[j][0]**2 + image[j][1]**2), "Input: Zero-Filled", axs[0])
        show_image(torch.sqrt(target[j][0]**2 + target[j][1]**2), "Target: Fully Sampled", axs[1])
        show_image(mask[j][0], "Mask", axs[2])
        plt.tight_layout()
        plt.show()

In [ ]:
print("Validation Loader:\nRunning 3 iterations with batch size 4...")
for i, batch in enumerate(valloader):
    if i >= 3:
        break
    print(f"\nBatch {i+1}")
    image, target, mean, std, norm, fname, slice_num, max_val, mask, masked_kspace = batch
    print(f"  image.shape: {image.shape}")
    print(f"  target.shape: {target.shape}")
    print(f"  mask.shape: {mask.shape}")
    print(f"  masked_kspace.shape: {masked_kspace.shape}")
    for j in range(4):
        print(f"  → Slice {slice_num[j]} | File: {fname[j]}")
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        def show_image(tensor_img, title, ax):
            img = tensor_img.numpy()
            img = np.abs(img)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            ax.imshow(img, cmap='gray')
            ax.set_title(title)
            ax.axis('off')
        show_image(torch.sqrt(image[j][0]**2 + image[j][1]**2), "Input: Zero-Filled", axs[0])
        show_image(torch.sqrt(target[j][0]**2 + target[j][1]**2), "Target: Fully Sampled", axs[1])
        show_image(mask[j][0], "Mask", axs[2])
        plt.tight_layout()
        plt.show()

In [ ]:
def complex_to_magnitude(x):
    # (batch_size, 2, H, W)
    return torch.sqrt(x[:, 0] ** 2 + x[:, 1] ** 2).unsqueeze(1)  # (batch_size, 1, H, W)